In [ ]:

 
import os

print(os.getcwd()) 
from dotenv import load_dotenv
import os
load_dotenv()
import pandas as pd
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
# CONNECTION AVEC LA BASE DE DONNÉES

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")  

engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

query = """
SELECT *
FROM ml_schema.obt_avito;
"""

df = pd.read_sql(query, engine)

print(df.head())
print(df.shape)

c:\Users\user\Desktop\machine_learning
   avito_id                                              titre  \
0         1            Appartement à vendre 154 m² à Marrakech   
1         2   Appartement à vendre 140 m² la marina Casablanca   
2         3             Appartement à vendre 61 m² à Marrakech   
3         4                      Appartement à vendre à Temara   
4         5  Appartement en vente 61m² à Gueliz Piscine Roo...   

                                                lien  \
0  https://www.avito.ma/fr/route_de_casablanca/ap...   
1  https://www.avito.ma/fr/marina/appartements/Ap...   
2  https://www.avito.ma/fr/gu%C3%A9liz/appartemen...   
3  https://www.avito.ma/fr/autre_secteur/appartem...   
4  https://www.avito.ma/fr/gu%C3%A9liz/appartemen...   

                          ville     prix  surface  chambres  salle_de_bain  \
0   Appartements dans Marrakech  2475000      154         3              3   
1  Appartements dans Casablanca  3900000      155         2            

In [17]:
print(df.columns)


Index(['avito_id', 'titre', 'lien', 'ville', 'prix', 'surface', 'chambres',
       'salle_de_bain', 'prix_log', 'prix_par_m2'],
      dtype='str')


# Extraction des données OBT / Développement des modèles ML 

In [31]:
# Charger les données depuis PostgreSQL
query = "SELECT * FROM ml_schema.obt_avito;"
df = pd.read_sql(query, engine)

# Variable cible (target)
y = df["prix"]

# Features
X = df.drop(columns=["prix", "id"], errors="ignore")

# Features numériques
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

# Features catégoriques
categorical_features = X.select_dtypes(
    include=["object", "string"]
).columns

# Pipeline numérique
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Pipeline catégorique
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Préprocessing global
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

# Pipeline ML complet
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Entraînement
model.fit(X_train, y_train)

# Prédiction
pred = model.predict(X_test)

# Évaluation
print("MAE:", mean_absolute_error(y_test, pred))
print("MSE:", mean_squared_error(y_test, pred))
print("R²:", r2_score(y_test, pred))

MAE: 67681.85440298507
MSE: 25552423521.33699
R²: 0.9553789996646889


In [33]:

# Modèle de Classification


# Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

# Metrics
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# SMOTE
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


# 1. Création de la variable cible


df["cible"] = df["prix_log"].apply(
    lambda x: "haut"
    if x >= 15
    else ("moyen" if x >= 14 else "bas")
)


# 2. Sélection des features et de la cible


X = df.drop(
    columns=[
        "avito_id",
        "titre",
        "lien",
        "prix_log",
        "cible",
        "prix",
        "prix_par_m2"
    ],
    errors="ignore"
)

y = df["cible"]


# 3. train-test split   


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# 4. Prétraitement des données 


num_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

cat_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

# 5. Plusieurs modèles


models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=1000
    )
}


# 6. Entraînement + Évaluation


for name, classifier in models.items():

    print("\n" + "="*50)
    print(f"MODEL : {name}")
    print("="*50)

    pipeline = ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("classifier", classifier)
    ])

    # Training
    print(df["cible"].value_counts())

    pipeline.fit(X_train, y_train)

    # Prediction
    y_pred = pipeline.predict(X_test)

    # Probabilities
    y_proba = pipeline.predict_proba(X_test)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted"
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted"
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted"
    )

    roc_auc = roc_auc_score(
        pd.get_dummies(y_test),
        y_proba,
        multi_class="ovr",
        average="weighted"
    )

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-score  : {f1:.4f}")
    print(f"ROC-AUC   : {roc_auc:.4f}")

C:\Users\user\AppData\Local\Temp\ipykernel_18560\660990829.py:72: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(



MODEL : Random Forest
cible
moyen    364
bas      207
haut      98
Name: count, dtype: int64

Classification Report:
              precision    recall  f1-score   support

         bas       0.76      0.76      0.76        41
        haut       0.64      0.70      0.67        20
       moyen       0.77      0.75      0.76        73

    accuracy                           0.75       134
   macro avg       0.72      0.74      0.73       134
weighted avg       0.75      0.75      0.75       134

Accuracy  : 0.7463
Precision : 0.7483
Recall    : 0.7463
F1-score  : 0.7470
ROC-AUC   : 0.8664

MODEL : Decision Tree
cible
moyen    364
bas      207
haut      98
Name: count, dtype: int64

Classification Report:
              precision    recall  f1-score   support

         bas       0.71      0.78      0.74        41
        haut       0.62      0.65      0.63        20
       moyen       0.78      0.73      0.75        73

    accuracy                           0.73       134
   macro avg    

In [20]:
model_pipelines = {}

for name, classifier in models.items():

    print("\n" + "="*50)
    print(f"MODEL : {name}")
    print("="*50)

    pipeline = ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("classifier", classifier)
    ])

    # Training
    pipeline.fit(X_train, y_train)

    # Save model
    model_pipelines[name] = pipeline

    # Prediction
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)

    print(classification_report(y_test, y_pred))


MODEL : Random Forest
              precision    recall  f1-score   support

         bas       0.76      0.76      0.76        41
        haut       0.64      0.70      0.67        20
       moyen       0.77      0.75      0.76        73

    accuracy                           0.75       134
   macro avg       0.72      0.74      0.73       134
weighted avg       0.75      0.75      0.75       134


MODEL : Decision Tree
              precision    recall  f1-score   support

         bas       0.71      0.78      0.74        41
        haut       0.62      0.65      0.63        20
       moyen       0.78      0.73      0.75        73

    accuracy                           0.73       134
   macro avg       0.70      0.72      0.71       134
weighted avg       0.73      0.73      0.73       134


MODEL : Logistic Regression
              precision    recall  f1-score   support

         bas       0.62      0.85      0.72        41
        haut       0.57      0.80      0.67        20


In [21]:
rf_pipeline = model_pipelines["Random Forest"]

feature_names = rf_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

importance = rf_pipeline.named_steps[
    "classifier"
].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(importance_df.head(15))

                                        Feature  Importance
0                                  num__surface    0.509344
1                                 num__chambres    0.118550
2                            num__salle_de_bain    0.083270
14      cat__ville_Appartements dans Casablanca    0.038196
27         cat__ville_Appartements dans Kénitra    0.032167
31       cat__ville_Appartements dans Marrakech    0.028791
47          cat__ville_Appartements dans Tanger    0.021402
39           cat__ville_Appartements dans Rabat    0.021086
48          cat__ville_Appartements dans Temara    0.012571
24             cat__ville_Appartements dans Fès    0.011590
11        cat__ville_Appartements dans Bouznika    0.009504
3           cat__ville_Appartements dans Agadir    0.008941
45         cat__ville_Appartements dans Tamaris    0.008687
42  cat__ville_Appartements dans Sidi Bouknadel    0.007916
38           cat__ville_Appartements dans Oujda    0.006456


In [22]:
errors = pd.DataFrame({
    "Réel": y_test,
    "Prédit": y_pred
})

wrong_predictions = errors[
    errors["Réel"] != errors["Prédit"]
]

print(wrong_predictions.head(20))
print("Nombre erreurs :", len(wrong_predictions))

      Réel Prédit
436  moyen   haut
39   moyen    bas
474   haut  moyen
653  moyen   haut
604  moyen   haut
422  moyen    bas
202    bas  moyen
17   moyen   haut
328    bas  moyen
597  moyen   haut
143  moyen   haut
272  moyen   haut
148  moyen    bas
319  moyen    bas
400  moyen    bas
506  moyen   haut
293    bas  moyen
263  moyen    bas
496  moyen    bas
481  moyen    bas
Nombre erreurs : 43


In [23]:
results = []

for name, classifier in models.items():

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy":
            accuracy_score(y_test, y_pred),

        "F1":
            f1_score(
                y_test,
                y_pred,
                average="weighted"
            )
    })

results_df = pd.DataFrame(results)

print(
    results_df.sort_values(
        by="F1",
        ascending=False
    )
)

                 Model  Accuracy        F1
0        Random Forest  0.679104  0.674632
1        Decision Tree  0.679104  0.674632
2  Logistic Regression  0.679104  0.674632
